# Experiment driver
Thin notebook: all logic lives in the package. Clone, install, run.

In [ ]:
# bootstrap (edit URL after you push the repo)
REPO = "https://github.com/josepheisner54/RLDB_Game"
!git clone $REPO repo -q && pip install -e repo -q
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from deckbuilder import *
QUICK = False
GPU = torch.cuda.is_available()
BATCH = 64 if QUICK else (2048 if GPU else 512)
STEPS = 40 if QUICK else (4000 if GPU else 900)

In [ ]:
policy, V, hist = train_combat(steps=STEPS, B=BATCH, eval_every=max(STEPS // 9, 1))
V = refine_value(policy, V, steps=10 if QUICK else 300)

In [ ]:
meta, _ = train_meta(V, steps=30 if QUICK else 500, B=BATCH * 2)
for name, d in [("never", None), ("random", "random"), ("meta", meta)]:
    r = simulate_runs(policy, d, B=2000)
    print(f"{name:>7}: run win {r['win']:.3f}   final HP {r['final_hp']:.1f}")

In [ ]:
# behavioral probes
r = simulate_runs(policy, meta, B=2000, record=True)
hp = torch.cat(r["camp_hp"]); rest = torch.cat(r["camp_rest"])
for lo, hi in [(1, 30), (30, 45), (45, 70)]:
    m = (hp >= lo) & (hp < hi)
    if m.sum() > 20:
        print(f"P(rest | HP {lo}-{hi}) = {rest[m].mean():.2f}")
u = r["upgrade"] / r["upgrade"].sum()
print("campfire:", {n: round(v, 2) for n, v in
      sorted(zip(S.NAMES[:S.M] + ["REST"], u.tolist()), key=lambda x: -x[1])[:5]})

In [ ]:
# checkpoints -> Drive (gitignored in the repo)
# from google.colab import drive; drive.mount('/content/drive')
# torch.save({"policy": policy.state_dict(), "V": V.state_dict(),
#             "meta": meta.state_dict()}, "/content/drive/MyDrive/deckbuilder_ckpt.pt")